## Training Simple Model

In [ ]:
import os

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer


import pandas as pd

main_dir = os.path.dirname(os.getcwd())

In [ ]:
path_offset_ary = pd.read_csv(
    os.path.join(main_dir, "training_data", "link_path_offset.csv"),
    header=0,
    sep="\t",
).values

In [ ]:
summary_dump_file = os.path.join(main_dir, "training_data", "summary.csv")

In [ ]:
class SentenceDataset(Dataset):
    def __init__(self, path_data, sum_file_path, max_length=128):
        self.path_data = path_data
        self.sum_file = open(sum_file_path, "br")
        self.max_length = max_length

    def __len__(self):
        return self.path_data.shape[0]

    def __getitem__(self, idx):
        s_off, t_off, step = self.path_data[idx]

        self.sum_file.seek(s_off, 0)
        s_sen = str(self.sum_file.readline(), encoding="utf-8").split("\t")[-1]
        self.sum_file.seek(t_off, 0)
        t_sen = str(self.sum_file.readline(), encoding="utf-8").split("\t")[-1]

        return {
            "source_sen": s_sen,
            "target_sen": t_sen,
            "step": torch.tensor(step, dtype=torch.long),
        }

In [ ]:
sent_dataset = SentenceDataset(
    path_data=path_offset_ary,
    sum_file_path=summary_dump_file,
    max_length=128,
)
sent_dataloader = DataLoader(sent_dataset, batch_size=1024, shuffle=True)

In [ ]:
sentance_model = SentenceTransformer("sentence-transformers/all-distilroberta-v1")

In [ ]:
class ScorePredictor(pl.LightningModule):
    def __init__(
        self,
        sentance_model=sentance_model,
        input_dim=768,
        hidden1_dim=1024,
        hidden2_dim=256,
        output_dim=1,
        lr=1e-5,
    ):
        super(ScorePredictor, self).__init__()
        # Load the pretrained sentence-transformer model
        self.sentence_model = sentance_model

        # Freeze the sentence-transformer model
        for param in self.sentence_model.parameters():
            param.requires_grad = False

        # Define a simple feedforward network
        self.regressor = torch.nn.Sequential(
            torch.nn.Linear(input_dim * 4, input_dim * 2),
            torch.nn.ReLU(),
            torch.nn.Linear(input_dim * 2, hidden1_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden1_dim, hidden2_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden2_dim, output_dim),
        )

        self.lr = lr
        self.loss_fn = lambda yp, yt: torch.mean(torch.abs(1 / (1 + yp) - 1 / (1 + yt)))

    def forward(self, start_sentance, target_sentance):
        # Use the sentence-transformer to get embeddings
        with torch.no_grad():
            s_embed = self.sentence_model.encode(start_sentance, convert_to_tensor=True)
            t_embed = self.sentence_model.encode(
                target_sentance, convert_to_tensor=True
            )

        diff_embed = torch.abs(s_embed - t_embed)
        square_embed = torch.pow(s_embed - t_embed, 2)

        embeddings = torch.concat((s_embed, t_embed, diff_embed, square_embed), axis=1)
        # Pass the embeddings to the regressor
        val = self.regressor(embeddings)
        return 4 * torch.exp(-val)

    def training_step(self, batch):
        source_sentences = (batch["source_sen"],)
        target_sentences = (batch["target_sen"],)
        scores = batch["step"]

        # Forward pass
        predictions = self.forward(
            source_sentences,
            target_sentences,
        )

        # Compute loss
        loss = self.loss_fn(predictions.squeeze(-1), scores)
        self.log("train_loss", loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.regressor.parameters(), lr=self.lr)

In [ ]:
score_predictor = ScorePredictor()

In [ ]:
# Create a Trainer object
trainer = pl.Trainer(max_epochs=40)

# Fit the model
trainer.fit(score_predictor, sent_dataloader)

In [ ]:
sent_dataset[567]

In [ ]:
def get_score(loc):
    return score_predictor(
        [sent_dataset[loc]["source_sen"]], [sent_dataset[loc]["target_sen"]]
    ).item(), sent_dataset[loc]["step"].item()

In [ ]:
[get_score(34564 - i) for i in range(20)]